# Creating a Simple Agent with Tracing

In [1]:
import dotenv
import os

from openai import OpenAI

dotenv.load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print(
        """Error: OPENAI_API_KEY environment variable not set. Please copy the .env.template file as .env and fill it in.
    
    You can execute these commands in the terminal to get started:
    cp .env.template .env
    code .env
    """
    )

# Test OpenAI Access
print(
    OpenAI()
    .responses.create(
        model=os.environ["OPENAI_DEFAULT_MODEL"], input="Say: We are up and running!"
    )
    .output_text
)

We are up and running!


In [2]:
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent

Create a simple Nutrition Assistant Agent

In [3]:
nutrition_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful assistant giving out nutrition advice.
    You give concise answers.
    """,
)

Let's execute the Agent:

In [4]:
with trace("Simple Nutrition Agent"):
    result = await Runner.run(nutrition_agent, "How healthy are bananas?")

print(result)

RunResult:
- Last agent: Agent(name="Nutrition Assistant", ...)
- Final output (str):
    Overall, yes—bananas are healthy when eaten in moderation.
    
    Key points:
    - Nutrients: potassium, vitamin C, vitamin B6, fiber, small amount of magnesium and manganese
    - Benefits: supports heart health, digestion, and energy; low in fat and calories (about 90-110 kcal per medium)
    - Considerations: natural sugars; moderate for those watching carbs or blood sugar; not a protein source; can have pesticide residues if not washed/organic
    
    Practical tips:
    - Great as a snack or sliced on cereal/yogurt
    - Pair with protein (nuts, yogurt) for staying power
    - Freeze when ripe for smoothies or a frozen treat
- 2 new item(s)
- 1 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


Streaming the answer to the screen, token by token

In [4]:
response_stream = Runner.run_streamed(nutrition_agent, "How healthy are bananas?")

async for event in response_stream.stream_events():
    if event.type == "raw_response_event" and isinstance(
        event.data, ResponseTextDeltaEvent
    ):
        print(event.data.delta, end="", flush=True)

Bananas are healthy as part of a balanced diet. Pros: good source of potassium, vitamin B6, vitamin C, fiber, and quick energy. Medium banana ~105 calories, natural sugars, and resistant starch when unripe. Cons: portion control needed for diabetes or total calorie goals; ripe bananas have more sugar. Great as a snack, in smoothies, or with yogurt. Cautions: people with kidney disease may need limits on potassium; allergy is rare.

_Good Job!_

In [9]:
from agents import Agent

placement_trainer_agent = Agent(
    name="Placement Trainer",
    instructions="""
    You are a practical placement preparation coach. Help students prepare for job
    placements, including resumes, aptitude tests, technical interviews, HR rounds,
    communication, and job-search planning.

    Give concise, actionable answers in at most 5 bullet points. Tailor advice to
    the student's role, experience, and timeline when provided. If essential details
    are missing, ask one focused follow-up question. For practice questions, provide
    a clear answer and a brief explanation. Never invent company-specific facts or
    promise placement outcomes.
    """,
)


In [11]:
with trace("Simple Placement Trainer"):
    result = await Runner.run(placement_trainer_agent, "How should i prepare for interview in 30 days?")

print(result)

RunResult:
- Last agent: Agent(name="Placement Trainer", ...)
- Final output (str):
    - Draft a focused plan and materials (Day 1): update resume/LinkedIn, 2–3 STAR stories highlighting outcomes, 1–2-minute personal pitch, and list 6–8 core skills for your target role.
    
    - Technical/aptitude backbone (Days 2–15): lay out role-specific topics, practice 40–60 problems or questions (coding, analytics, or domain basics), plus 1–2 mock coding/tech problems per week; ensure you can explain concepts clearly.
    
    - Behavioral and mock interviews (Days 16–22): schedule 4–6 mock interviews (with friends/mentors or platforms), practice STAR responses, and refine your resume-presentation flow; record to improve body language and clarity.
    
    - Company-specific prep (Days 23–28): research top 4–6 target companies, prepare tailored questions, and do 2–3 full-length practice rounds mimicking interview cadence; prepare relatable success stories to align with each company.
    
    -

In [13]:
student_agent = Agent(
    name="student Assistant",
    instructions="""
    You are a helpful assistant giving out nutrition advice.
    You give concise answers.
    """,
)

In [14]:
with trace("Simple student Agent"):
    result = await Runner.run(student_agent, "How to prepare for exam?")

print(result)

RunResult:
- Last agent: Agent(name="student Assistant", ...)
- Final output (str):
    Here’s a quick, nutrition-focused prep guide for exams:
    
    - Sleep well: 7–9 hours, consistent schedule.
    - Pre-study meal: balanced mix of complex carbs, protein, and healthy fat (e.g., oats with yogurt and berries, or whole-grain toast with eggs).
    - During study: steady snacks every 2–3 hours to maintain energy.
      - Good options: fruit + nuts, yogurt + berries, hummus with veggies, whole-grain crackers with cheese.
    - Focus foods (brain-boosters): fatty fish (or flax/chia seeds), leafy greens, berries, nuts, eggs.
    - Hydration: steady water intake; limit sugary drinks.
    - Caffeine: moderate amounts, avoid late-day caffeine to not disrupt sleep.
    - Exam day meal: easy-to-ddigest carbs + protein (e.g., oatmeal with peanut butter, banana; or a yogurt smoothie with fruit).
    - Breaks: short 5–10 minute breaks; quick movement to stay alert.
    
    If you want, tell me y